# Proyecto Final: Predicción de Fuga de Clientes
El operador de telecomunicaciones **Interconnect** busca reducir la cancelación de clientes (`Churn`) mediante un sistema que identifique con anticipación a quienes podrían darse de baja, para ofrecerles promociones y planes especiales.

**Objetivo:** Desarrollar y comparar modelos capaces de predecir si un cliente se dará de baja próximamente (Sí/No).
- El modelo se evaluará principalmente con AUC-ROC y con Recall como métrica adicional.

## Etapa 1: Plan de Trabajo
*Escribe aquí tu plan de trabajo inicial. Aborda brevemente:*
1. *¿Cómo planeas unir los datos y qué harás con los valores nulos generados?*
### Respuesta: 
Uniré las 4 tablas de datos con merge usando la columna 'CustomerID', usando uniones izquierdas, con respecto a los valores nulos que se generen, seran nulos con proposito, el cual es señalar que determinado cliente no contrato cierto servicio, por lo tanto hay que conservarlos, planeo ponerle una etiqueta categorica que indique que no tiene el servicio

2. *¿Cuál será tu variable objetivo y qué tipo de problema de Machine Learning resolverás?*
### Respuesta:
La variable objetivo será el estado de cancelación del cliente (`Churn`), la cual definiré como una variable binaria (1 si el cliente canceló el servicio y 0 si continúa activo), creada a partir de la columna de fecha de finalización del contrato (`EndDate`). Por lo tanto, el problema que resolveremos es de **Clasificación Binaria**.

3. *¿Qué pasos de preprocesamiento (codificación categórica, fechas, etc.) consideras necesarios y sobre que variables?*
### Respuesta:
Realizaremos los siguientes pasos de preprocesamiento:
* **Fechas:** Transformaremos las columnas `BeginDate` y `EndDate` para calcular la antigüedad del cliente en días o meses (útil para que el modelo entienda cuánto tiempo lleva con la empresa).
* **Codificación Categórica:** Aplicaremos técnicas como One-Hot Encoding (OHE) a las variables categóricas (como el tipo de contrato, método de pago, género, etc.) para convertirlas en formato numérico apto para los modelos.
* **Escalado:** Estandarizaremos las variables numéricas (como la antigüedad y los cargos mensuales) para que tengan la misma escala.*

4. *¿Qué modelos planeas entrenar?*
### Respuesta:
Planeamos entrenar al menos dos tipos de modelos diferentes para comparar su rendimiento:
* **Modelo Baseline:** Una **Regresión Logística** como punto de partida sencillo.
* **Modelo Avanzado / Basado en Árboles:** Un modelo como **Random Forest** o **LightGBM** para capturar patrones más complejos en los datos.
Ambos modelos serán evaluados principalmente con la métrica **AUC-ROC** (buscando superar el objetivo de 0.88) y **Recall** como métrica de apoyo.

## Etapa 2: Código de Solución

### 1. Exploración de Datos (EDA)
*Carga de datos, análisis de distribuciones, identificación de anomalías.*

### 2. Preprocesamiento e Ingeniería de Características
*Procesar valores nulos, creación de la variable objetivo, codificación de variables categóricas (justifica tu elección de método).*
*Pista: Los modelos predictivos no entienden de fechas en formato texto. ¿Cómo puedes transformar las fechas de inicio y fin en una variable numérica útil para el modelo?*

### 3. Selección de Variables y Entrenamiento de Modelos (Baseline)
*Entrena al menos dos modelos distintos sin aplicar técnicas de balanceo de clases. Evalúa su AUC-ROC y Recall.*

### 4. Optimización y Manejo de Desbalance
*Aplica al menos una técnica para manejar el desbalance de clases (upsampling, downsampling, o ajuste de pesos) y busca los mejores hiperparámetros. Evalúa nuevamente.*

## Etapa 3: Informe de Solución
*Escribe aquí tu informe final para el equipo de negocio. Asegúrate de responder:*
1. *¿Qué modelo elegiste finalmente y por qué?*
2. *¿Cuáles fueron las métricas finales (AUC-ROC y Recall) en el conjunto de prueba?*
3. *En términos de negocio: ¿Qué significa tu valor de Recall? ¿Cómo impactaría tu modelo en la retención de clientes si el equipo de marketing lo utiliza hoy?*

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.metrics import recall_score, classification_report

## Etapa 2: Código de Solución

### 1. Exploración de Datos (EDA)
*Carga de datos, análisis de distribuciones, identificación de anomalías.*

### 2. Preprocesamiento e Ingeniería de Características
*Procesar valores nulos, creación de la variable objetivo, codificación de variables categóricas (justifica tu elección de método).*
*Pista: Los modelos predictivos no entienden de fechas en formato texto. ¿Cómo puedes transformar las fechas de inicio y fin en una variable numérica útil para el modelo?*

### 3. Selección de Variables y Entrenamiento de Modelos (Baseline)
*Entrena al menos dos modelos distintos sin aplicar técnicas de balanceo de clases. Evalúa su AUC-ROC y Recall.*

### 4. Optimización y Manejo de Desbalance
*Aplica al menos una técnica para manejar el desbalance de clases (upsampling, downsampling, o ajuste de pesos) y busca los mejores hiperparámetros. Evalúa nuevamente.*

In [2]:
#función de carga y exploracion de los datos
def load_view_data(data):
    df = pd.read_csv(data)
    print('INFO DE LOS DATOS')
    print(df.info())
    print()
    print('DESCRIPCION DE LOS DATOS')
    print(df.describe())
    print()
    print('VALORES NULOS DE LOS DATOS')
    print(df.isna().sum())
    return df

contract = load_view_data('/datasets/final_provider/contract.csv')

INFO DE LOS DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB
None

DESCRIPCION DE LOS DATOS
       MonthlyCharges
count     7043.000000
mean        64.761692
std         30.090047
min         18.250000
25%         35.500000
50%         70.350000
75%         89.850000
max        118.750000

VALORES NULOS DE LOS DATOS
customerID          0
BeginDate           0
EndDate             0
Type                0
Paperle

In [3]:
personal = load_view_data('/datasets/final_provider/personal.csv')

INFO DE LOS DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     7043 non-null   object
 1   gender         7043 non-null   object
 2   SeniorCitizen  7043 non-null   int64 
 3   Partner        7043 non-null   object
 4   Dependents     7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB
None

DESCRIPCION DE LOS DATOS
       SeniorCitizen
count    7043.000000
mean        0.162147
std         0.368612
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000

VALORES NULOS DE LOS DATOS
customerID       0
gender           0
SeniorCitizen    0
Partner          0
Dependents       0
dtype: int64


In [4]:
internet = load_view_data('/datasets/final_provider/internet.csv')

INFO DE LOS DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5517 entries, 0 to 5516
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        5517 non-null   object
 1   InternetService   5517 non-null   object
 2   OnlineSecurity    5517 non-null   object
 3   OnlineBackup      5517 non-null   object
 4   DeviceProtection  5517 non-null   object
 5   TechSupport       5517 non-null   object
 6   StreamingTV       5517 non-null   object
 7   StreamingMovies   5517 non-null   object
dtypes: object(8)
memory usage: 344.9+ KB
None

DESCRIPCION DE LOS DATOS
        customerID InternetService OnlineSecurity OnlineBackup  \
count         5517            5517           5517         5517   
unique        5517               2              2            2   
top     8085-MSNLK     Fiber optic             No           No   
freq             1            3096           3498         3088   

       Devic

In [5]:
phone = load_view_data('/datasets/final_provider/phone.csv')

INFO DE LOS DATOS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6361 entries, 0 to 6360
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     6361 non-null   object
 1   MultipleLines  6361 non-null   object
dtypes: object(2)
memory usage: 99.5+ KB
None

DESCRIPCION DE LOS DATOS
        customerID MultipleLines
count         6361          6361
unique        6361             2
top     4282-MSACW            No
freq             1          3390

VALORES NULOS DE LOS DATOS
customerID       0
MultipleLines    0
dtype: int64


In [6]:
df_merged = contract.merge(personal, on='customerID', how='left') \
                    .merge(internet, on='customerID', how='left') \
                    .merge(phone, on='customerID', how='left')
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
 8   gender            7043 non-null   object 
 9   SeniorCitizen     7043 non-null   int64  
 10  Partner           7043 non-null   object 
 11  Dependents        7043 non-null   object 
 12  InternetService   5517 non-null   object 
 13  OnlineSecurity    5517 non-null   object 
 14  OnlineBackup      5517 non-null   object 
 15  DeviceProtection  5517 non-null   object 
 16  TechSupport       5517 non-null   object 


In [7]:
cols_internet = ['InternetService', 'OnlineSecurity', 'OnlineBackup', 
                 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_merged[cols_internet] = df_merged[cols_internet].fillna('No')
df_merged['MultipleLines'] = df_merged['MultipleLines'].fillna('No')
df_merged['TotalCharges'] = pd.to_numeric(df_merged['TotalCharges'], errors='coerce')
df_merged['TotalCharges'] = df_merged['TotalCharges'].fillna(0)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   float64
 8   gender            7043 non-null   object 
 9   SeniorCitizen     7043 non-null   int64  
 10  Partner           7043 non-null   object 
 11  Dependents        7043 non-null   object 
 12  InternetService   7043 non-null   object 
 13  OnlineSecurity    7043 non-null   object 
 14  OnlineBackup      7043 non-null   object 
 15  DeviceProtection  7043 non-null   object 
 16  TechSupport       7043 non-null   object 


In [8]:
df_merged['BeginDate'] = pd.to_datetime(df_merged['BeginDate'])
df_merged['Churn'] = (df_merged['EndDate'] != 'No').astype(int)
df_merged['EndDate'] = pd.to_datetime(df_merged['EndDate'], errors='coerce')
df_merged['Churn'].value_counts()

0    5174
1    1869
Name: Churn, dtype: int64

In [9]:

fecha_referencia = df_merged['BeginDate'].max()
fecha_fin_efectiva = df_merged['EndDate'].fillna(fecha_referencia)

df_merged['ClientAge'] = (fecha_fin_efectiva - df_merged['BeginDate']).dt.days

df_merged[['BeginDate', 'EndDate', 'Churn', 'ClientAge']].head()


,BeginDate,EndDate,Churn,ClientAge
0,2020-01-01,NaT,0,31
1,2017-04-01,NaT,0,1036
2,2019-10-01,2019-12-01,1,61
3,2016-05-01,NaT,0,1371
4,2019-09-01,2019-11-01,1,61


In [10]:
df_merged = df_merged.drop(['BeginDate', 'EndDate', 'customerID'], axis=1)
df_merged = pd.get_dummies(df_merged, drop_first=True)
df_merged.shape

(7043, 23)

## Resumen de esta sección

* *Cree una función para simplificar el proceso de carga y exploración inicial de los datos*
* *Uní las 4 tablas de datos mediante la columna "customerID"*
* *Esta unión generó valores nulos ya previstos y explicados al inicio del proyecto, y fueron rellenados con un "No"*
* *Convertí las columnas a sus formatos adecuados, y creé la variable objetivo "Churn"*
* *Creé una columna que calcule la antiguedad del cliente, y eliminé las columnas innecesarias para el entrenamiento*
* *Eliminé la columna customerID para evitar que la codificación One-Hot creara miles de columnas innecesarias, y luego usé OHE para convertir los datos de texto a numéricos*

In [11]:
features = df_merged.drop('Churn', axis=1)
target = df_merged['Churn']

features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.40, random_state=12345, stratify=target
)

features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.50, random_state=12345, stratify=target_temp
)

print("Entrenamiento:", features_train.shape)
print("Validación:", features_valid.shape)
print("Prueba:", features_test.shape)

Entrenamiento: (4225, 22)
Validación: (1409, 22)
Prueba: (1409, 22)


In [12]:
pd.options.mode.chained_assignment = None

numeric_features = ['MonthlyCharges', 'TotalCharges', 'ClientAge']
scaler = StandardScaler()

features_train.loc[:, numeric_features] = scaler.fit_transform(features_train[numeric_features])
features_valid.loc[:, numeric_features] = scaler.transform(features_valid[numeric_features])
features_test.loc[:, numeric_features] = scaler.transform(features_test[numeric_features])

pd.options.mode.chained_assignment = 'warn'

In [13]:
model_lr = LogisticRegression(random_state=12345, max_iter=1000)
model_lr.fit(features_train, target_train)

model_rf = RandomForestClassifier(random_state=12345)
model_rf.fit(features_train, target_train)

model_lgb = LGBMClassifier(random_state=12345, verbosity=-1)
model_lgb.fit(features_train, target_train)

print("¡Modelos baseline entrenados con éxito!")

¡Modelos baseline entrenados con éxito!


In [14]:
probabilities_valid_lr = model_lr.predict_proba(features_valid)[:, 1]
preds_valid_lr = model_lr.predict(features_valid)

auc_lr = roc_auc_score(target_valid, probabilities_valid_lr)
recall_lr = recall_score(target_valid, preds_valid_lr)

print(f"ROC-AUC de Logistic Regression: {auc_lr:.4f}")
print(f"Recall de Logistic Regression:  {recall_lr:.4f}")

ROC-AUC de Logistic Regression: 0.8356
Recall de Logistic Regression:  0.5802


In [15]:
probabilities_valid_rf = model_rf.predict_proba(features_valid)[:, 1]
preds_valid_rf = model_rf.predict(features_valid)

auc_rf = roc_auc_score(target_valid, probabilities_valid_rf)
recall_rf = recall_score(target_valid, preds_valid_rf)

print(f"ROC-AUC de Random Forest: {auc_rf:.4f}")
print(f"Recall de Random Forest:  {recall_rf:.4f}")

ROC-AUC de Random Forest: 0.8336
Recall de Random Forest:  0.5348


In [16]:
probabilities_valid_lgb = model_lgb.predict_proba(features_valid)[:, 1]
preds_valid_lgb = model_lgb.predict(features_valid)

auc_lgb = roc_auc_score(target_valid, probabilities_valid_lgb)
recall_lgb = recall_score(target_valid, preds_valid_lgb)

print(f"ROC-AUC de LightGBM: {auc_lgb:.4f}")
print(f"Recall de LightGBM:  {recall_lgb:.4f}")

ROC-AUC de LightGBM: 0.8754
Recall de LightGBM:  0.6096


In [17]:
learning_rates = [0.05, 0.1]
n_estimators_list = [100, 200]
max_depth_list = [5, 10, -1]
num_leaves_list = [31, 63]

mejor_auc = 0
mejor_modelo = None
mejores_params = {}

print("Iniciando búsqueda optimizada de hiperparámetros...\n")

for lr in learning_rates:
    for n_est in n_estimators_list:
        for depth in max_depth_list:
            for leaves in num_leaves_list:
                
                # Validar coherencia lógica en LightGBM
                if depth > 0 and leaves >= (2 ** depth):
                    continue

                modelo_prueba = LGBMClassifier(
                    learning_rate=lr,
                    n_estimators=n_est,
                    max_depth=depth,
                    num_leaves=leaves,
                    random_state=12345,
                    verbosity=-1
                )
                
                modelo_prueba.fit(features_train, target_train)
                
                probs = modelo_prueba.predict_proba(features_valid)[:, 1]
                auc_actual = roc_auc_score(target_valid, probs)
                
                if auc_actual > mejor_auc:
                    mejor_auc = auc_actual
                    mejor_modelo = modelo_prueba
                    mejores_params = {
                        'learning_rate': lr,
                        'n_estimators': n_est,
                        'max_depth': depth,
                        'num_leaves': leaves
                    }

print("¡Búsqueda finalizada con éxito!")
print(f"Mejor ROC-AUC obtenido: {mejor_auc:.4f}")
print("Mejores hiperparámetros:", mejores_params)

Iniciando búsqueda optimizada de hiperparámetros...

¡Búsqueda finalizada con éxito!
Mejor ROC-AUC obtenido: 0.8770
Mejores hiperparámetros: {'learning_rate': 0.1, 'n_estimators': 200, 'max_depth': 5, 'num_leaves': 31}


In [18]:
scale_weight = 5174 / 1869

model_final = LGBMClassifier(
    learning_rate=0.1,
    n_estimators=200,
    max_depth=5,
    num_leaves=31,
    scale_pos_weight=scale_weight,
    random_state=12345,
    verbosity=-1
)

model_final.fit(features_train, target_train)

probabilities_valid_lgb = model_final.predict_proba(features_valid)[:, 1]
auc_lgb = roc_auc_score(target_valid, probabilities_valid_lgb)
print(f"ROC-AUC de LightGBM Optimizado: {auc_lgb:.4f}")

ROC-AUC de LightGBM Optimizado: 0.8875


In [19]:
probs_test = model_final.predict_proba(features_test)[:, 1]
preds_test = model_final.predict(features_test)

auc_test = roc_auc_score(target_test, probs_test)
recall_test = recall_score(target_test, preds_test)

print(f"=== RESULTADOS FINALES EN CONJUNTO DE PRUEBA ===")
print(f"AUC-ROC Final: {auc_test:.4f}")
print(f"Recall Final:  {recall_test:.4f}")
print("\nReporte de Clasificación Completo:\n")
print(classification_report(target_test, preds_test))

=== RESULTADOS FINALES EN CONJUNTO DE PRUEBA ===
AUC-ROC Final: 0.8909
Recall Final:  0.7674

Reporte de Clasificación Completo:

              precision    recall  f1-score   support

           0       0.91      0.86      0.88      1035
           1       0.66      0.77      0.71       374

    accuracy                           0.83      1409
   macro avg       0.79      0.81      0.80      1409
weighted avg       0.84      0.83      0.84      1409



# *CONCLUSIONES*

Elegí el modelo de LightGBM porque fue el de mejor resultado AUC-SCORE y mayor recall, alcanzando un AUC de 0.87 (Excelente) y un recall de 0.60 con hiperparámetros en bruto. Por medio de un bucle de búsqueda de hiperparámetros y ajuste de pesos, logré optimizarlo para alcanzar un AUC de 0.89 y aumentar significativamente el recall hasta 0.76. A continuación dejo una breve explicación sobre estas métricas:

* **AUC-SCORE:** Qué tan bien mi modelo distingue y predice qué clientes se irán y quiénes no.
* **RECALL:** Mide la capacidad de detección de la clase positiva; un recall de 0.76 significa que de cada 100 clientes que realmente decidieron cancelar el servicio, mi modelo logró identificar correctamente a 76 de ellos antes de que se fueran.

Usando mi modelo en este momento, el equipo de marketing podría predecir qué clientes tienen riesgo de fuga para ofrecerles incentivos de retención, logrando detectar de manera efectiva al 76% de los clientes que verdaderamente iban a cancelar.